# **Problem Statement**
---



**Writing Viterbi Algorithm for the Primer**

Write the Viterbi algorithm to implment Nature Primer.

Here are some suggestions:

a. You can begin by first defining all the parameters, such as states, transition matrix, and emmision matrix etc.

b. You can write a function to exactly calculate the values mentioned in the primer, for example, you can define a function get_log_prob_of_a_given_path ("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA"). This should output -41.22.

By doing the above two, you earn 1 mark.

Now, you have to implement this in real to get max likely path that would emmit the observed sequence. You MUST note that maximum likely path will just be Es, but that is okay. Implementation is the key.

# **Approach**


---




####  1. Model the Problem

We begin by defining three components of the HMM:

- **States**: These represent biological regions like exon (E), donor splice site (5), and intron (I).
- **Transition Probabilities**: These give the chance of moving from one state to another (e.g., from exon to splice site).
- **Emission Probabilities**: These define the likelihood of emitting a nucleotide (A, C, G, T) from each state.
- **Initial Probabilities**: These indicate where the sequence is most likely to begin.

These probabilities are either given or assumed based on biological knowledge (e.g., exon regions emit nucleotides equally).

\


####  2. Calculate Log Probability of a Known Path

To validate the model setup, we first write a function to compute the **log-probability of a known path emitting a given sequence**.

- For each position in the sequence:
  - Take the transition probability from the previous state to the current one.
  - Take the emission probability of the observed nucleotide in the current state.
  - Multiply these and take the logarithm.
- Add up all the log-values.
- If the last state is an intron, include the log-probability of transitioning to the end state.

This gives the overall log-probability of that known state path emitting the observed sequence.


\

####  3. Implement the Viterbi Algorithm

Next, we use the Viterbi algorithm to find the **most likely hidden path** without knowing it in advance.

The main idea is to use **dynamic programming**:

- We create a matrix where each row is a state and each column is a position in the DNA sequence.
- Each cell stores the **maximum log-probability** of reaching that state at that position.
- For each step in the sequence:
  - We look at all possible previous states.
  - For each, calculate the total log-probability to the current state.
  - Pick the path with the highest probability.
- We also store **backpointers** to trace the best path backward at the end.

After processing all positions, we identify the final state with the highest probability and backtrack using the pointers to reconstruct the best state sequence.

---

Using this structured approach, we are able to:

- Validate our model with known paths.
- Apply the Viterbi algorithm to **infer the hidden biological regions** from the observed nucleotide sequence.



In [2]:
import numpy as np
import math

# Defining HMM Parameters:
states = ['E', '5', 'I']
nucleotides = ['A', 'C', 'G', 'T']

# 2. Initial Probabilities: Probability of starting with an E or an I
initial_probabilities = {'E': 1.0, '5': 0.0, 'I': 0.0}

# 3. Transition matrix: Probabilities of moving from one state to another
transition_probabilities = {
    'Start': {'E': 1.0, '5': 0.0, 'I': 0.0, 'End': 0.0},
    'E': {'E': 0.9, '5': 0.1, 'I': 0.0, 'End': 0.0},
    '5': {'E': 0.0, '5': 0.0, 'I': 1.0, 'End': 0.0},
    'I': {'E': 0.0, '5': 0.0, 'I': 0.9, 'End': 0.1}
}

# 4. Emission probabilities: Probability of emitting nucleotides (A, C, G, T) in each state
emission_probs = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.00, 'G': 0.95, 'T': 0.00},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4},
}
# Log function to avoid taking log of zero
def log(x):
    if x == 0:
        return -math.inf
    else:
        return math.log(x)

# Log probability of given path
def get_log_prob_of_a_given_path(state_path, observed_sequence):
    log_prob = 0.0
    if len(state_path) != len(observed_sequence):
        raise ValueError("The length of state path and the observed sequence must be the same")

    prev_state = 'Start'
    for i in range(len(observed_sequence)):
        current_state = state_path[i]
        observed_state = observed_sequence[i]
        log_prob += log(transition_probabilities[prev_state][current_state]) + \
                    log(emission_probs[current_state][observed_state])
        prev_state = current_state
    if prev_state == 'I':
        log_prob += log(transition_probabilities[prev_state]['End'])

    return log_prob

# Example State Path and Observed Sequence
state_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
observed_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
ans = get_log_prob_of_a_given_path(state_path, observed_sequence)
print(f"Log probability of the given state path: {ans}")

Log probability of the given state path: -41.21967768602254


In [8]:
# Viterbi Algorithm Implementation
def viterbiAlgo(observed_sequence):
    num_states = len(states)
    num_observations = len(observed_sequence)
    viterbi_matrix = np.full((num_states, num_observations), -np.inf)
    backpointers = np.zeros((num_states, num_observations), dtype=int)

    # Convert to state indices
    state_to_index = {s: i for i, s in enumerate(states)}

    # Initializing the DP table
    for i, state in enumerate(states):
        viterbi_matrix[i, 0] = log(initial_probabilities[state]) + \
                               log(emission_probs[state][observed_sequence[0]])

    # Recursive part
    for k in range(1, num_observations):
        for curr_idx, current_state in enumerate(states):
            max_log_prob = -np.inf
            best_prev_idx = 0
            for prev_idx, prev_state in enumerate(states):
                log_prob = viterbi_matrix[prev_idx, k - 1] + \
                           log(transition_probabilities[prev_state][current_state])
                if log_prob > max_log_prob:
                    max_log_prob = log_prob
                    best_prev_idx = prev_idx
            viterbi_matrix[curr_idx, k] = max_log_prob + \
                                          log(emission_probs[current_state][observed_sequence[k]])
            backpointers[curr_idx, k] = best_prev_idx

    # Retracing the best path
    best_path = []
    best_final_idx = np.argmax(viterbi_matrix[:, -1])
    best_path.append(states[best_final_idx])

    for i in range(num_observations - 1, 0, -1):
        best_final_idx = backpointers[best_final_idx, i]
        best_path.insert(0, states[best_final_idx])

    return best_path, np.max(viterbi_matrix[:, -1])

# Run Viterbi Algorithm on observed sequence
best_path, log_prob = viterbiAlgo(observed_sequence)
print(f"Most probable path is {best_path} with probability {log_prob}")

Most probable path is ['E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E'] with probability -38.677666280562796
